## Chapter 3: Coding Mechanisms

### 3.3 Attehnding to different parts of the input with self-attention

#### 3.3.1 A simple self-attention mechanism without trainable weights

In [22]:
import torch

inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your      (x^1)
     [0.55, 0.87, 0.66], # journey   (x^2)
     [0.57, 0.85, 0.64], # starts    (x^3)
     [0.22, 0.58, 0.33], # with      (x^4)
     [0.77, 0.25, 0.10], # one       (x^5)
     [0.05, 0.80, 0.55]] # step      (x^6)
)

In [23]:
input_query = inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [24]:
input_1 = inputs[0]
input_1

tensor([0.4300, 0.1500, 0.8900])

In [25]:
0.55 * 0.43 + 0.87 * 0.15 + 0.66 * 0.89

0.9544

In [26]:
torch.dot(input_query, input_1)

tensor(0.9544)

In [27]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * input_query[idx]

print(res)

tensor(0.9544)


In [28]:
i = 0

res = torch.dot(inputs[i], input_query)
res

tensor(0.9544)

In [29]:
inputs.shape[0]

6

In [30]:
query = inputs[i]

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, input_query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [31]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
attn_weights_2_tmp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [32]:
attn_weights_2_tmp.sum()

tensor(1.0000)

In [33]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

softmax_naive(attn_scores_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [34]:
attn_weights_2 = torch.softmax(attn_scores_2,  dim = 0)

In [35]:
query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape)
for i, x_i in  enumerate(inputs):
    print(f"{attn_weights_2[i]} ---> {x_i}")
    context_vec_2 += attn_weights_2[i] * x_i
    
print(context_vec_2)

0.13854756951332092 ---> tensor([0.4300, 0.1500, 0.8900])
0.2378913015127182 ---> tensor([0.5500, 0.8700, 0.6600])
0.23327402770519257 ---> tensor([0.5700, 0.8500, 0.6400])
0.12399158626794815 ---> tensor([0.2200, 0.5800, 0.3300])
0.10818186402320862 ---> tensor([0.7700, 0.2500, 0.1000])
0.15811361372470856 ---> tensor([0.0500, 0.8000, 0.5500])
tensor([0.4419, 0.6515, 0.5683])


In [36]:
for i, x_i in enumerate(inputs):
    print(x_i)

tensor([0.4300, 0.1500, 0.8900])
tensor([0.5500, 0.8700, 0.6600])
tensor([0.5700, 0.8500, 0.6400])
tensor([0.2200, 0.5800, 0.3300])
tensor([0.7700, 0.2500, 0.1000])
tensor([0.0500, 0.8000, 0.5500])


#### 3.3.2 Computing attention weights

In [46]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=1)
all_context_vecs = attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

### 3.4 Implementing self-attention with trainable weights

#### 3.4.1 Computing the attention weights step by step

In [58]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [62]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

In [63]:
W_query

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)

In [64]:
W_key

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True)

In [65]:
W_value

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]], requires_grad=True)

In [67]:
query_2 = x_2 @ W_query

query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [ ]:
keys = inputs @ W_key
value = inputs @ W_value

keys.shape

torch.Size([6, 2])

In [74]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

In [76]:
keys_2 = keys[1]
attn_score_22 = torch.dot(query_2, keys_2)

In [77]:
attn_score_22

tensor(1.8524, grad_fn=<DotBackward0>)

In [78]:
attn_scores_2 = query_2 @ keys.T
attn_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [80]:
d_k = keys.shape[1]

attn_weights_2 = torch.softmax(attn_scores_2 / d_k ** 0.5, dim = -1)
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [82]:
attn_weights_2.sum(dim=-1)

tensor(1., grad_fn=<SumBackward1>)

In [ ]:
context_vec_2 = attn_weights_2 @ value

context_vec_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

#### 3.4.2 Implementing a compact SelfAttention class

In [87]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        
    def forward(self, x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / d_k ** 0.5, dim = -1)
        context_vec = attn_weights @ values
        
        return context_vec
    
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
sa_v1(inputs)

tensor([[0.3507, 0.8808],
        [0.3566, 0.8973],
        [0.3563, 0.8966],
        [0.3464, 0.8692],
        [0.3446, 0.8644],
        [0.3502, 0.8795]], grad_fn=<MmBackward0>)

In [89]:
m = torch.nn.Linear(2, 3)
m.bias

Parameter containing:
tensor([ 0.2911,  0.5878, -0.0934], requires_grad=True)

In [ ]:
import torch.nn as nn

class CausalAttention(nn.Module):
    
    def __init__(self, d_in, d_out, qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        
    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / d_k ** 0.5, dim = -1)
        context_vec = attn_weights @ values
        
        return context_vec
    
torch.manual_seed(123)
ca = CausalAttention(d_in, d_out)
ca(inputs)

tensor([[-0.6448,  0.1061],
        [-0.6463,  0.1029],
        [-0.6463,  0.1029],
        [-0.6442,  0.1021],
        [-0.6442,  0.1036],
        [-0.6447,  0.1017]], grad_fn=<MmBackward0>)

### 3.5 Hiding future words with causal attention

#### 3.5.1 Applying a causal attention mask

In [92]:
# Your journey starts with one step

In [ ]:
# Your -> journey

In [ ]:
queries = ca.W_query(inputs)
keys = ca.W_key(inputs)
values = ca.W_value(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / d_k ** 0.5, dim = -1)

In [97]:
attn_weights

tensor([[0.1791, 0.1701, 0.1700, 0.1583, 0.1623, 0.1602],
        [0.1628, 0.1737, 0.1736, 0.1616, 0.1653, 0.1630],
        [0.1630, 0.1736, 0.1735, 0.1616, 0.1653, 0.1630],
        [0.1617, 0.1707, 0.1707, 0.1649, 0.1666, 0.1654],
        [0.1682, 0.1703, 0.1702, 0.1626, 0.1651, 0.1636],
        [0.1596, 0.1717, 0.1717, 0.1648, 0.1668, 0.1655]],
       grad_fn=<SoftmaxBackward0>)

In [100]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [102]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1791, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1628, 0.1737, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1630, 0.1736, 0.1735, 0.0000, 0.0000, 0.0000],
        [0.1617, 0.1707, 0.1707, 0.1649, 0.0000, 0.0000],
        [0.1682, 0.1703, 0.1702, 0.1626, 0.1651, 0.0000],
        [0.1596, 0.1717, 0.1717, 0.1648, 0.1668, 0.1655]],
       grad_fn=<MulBackward0>)


In [105]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4839, 0.5161, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3196, 0.3403, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2421, 0.2555, 0.2555, 0.2468, 0.0000, 0.0000],
        [0.2011, 0.2036, 0.2035, 0.1944, 0.1974, 0.0000],
        [0.1596, 0.1717, 0.1717, 0.1648, 0.1668, 0.1655]],
       grad_fn=<DivBackward0>)


In [107]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2477,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1301, 0.2214,   -inf,   -inf,   -inf,   -inf],
        [0.1311, 0.2198, 0.2194,   -inf,   -inf,   -inf],
        [0.0395, 0.1158, 0.1157, 0.0668,   -inf,   -inf],
        [0.1120, 0.1295, 0.1291, 0.0646, 0.0858,   -inf],
        [0.0365, 0.1395, 0.1394, 0.0821, 0.0989, 0.0875]],
       grad_fn=<MaskedFillBackward0>)


In [109]:
torch.exp(torch.tensor(float("-inf")))

tensor(0.)

In [111]:
attn_weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4839, 0.5161, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3196, 0.3403, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2421, 0.2555, 0.2555, 0.2468, 0.0000, 0.0000],
        [0.2011, 0.2036, 0.2035, 0.1944, 0.1974, 0.0000],
        [0.1596, 0.1717, 0.1717, 0.1648, 0.1668, 0.1655]],
       grad_fn=<SoftmaxBackward0>)


#### 3.5.2 Masking additional attention weights with dropout

In [115]:
torch.manual_seed(123)

layer = torch.nn.Dropout(0.5)

In [116]:
example = torch.ones(6, 6)

In [117]:
layer(example)

tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])

In [119]:
layer(attn_weights)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 1.0323, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6805, 0.6803, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5111, 0.5110, 0.0000, 0.0000, 0.0000],
        [0.4021, 0.4071, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3433, 0.3433, 0.0000, 0.3336, 0.3309]],
       grad_fn=<MulBackward0>)

#### 3.5.3 Implementing a compact causal self-attention class

In [122]:
batch = torch.stack((inputs, inputs), dim=0)
batch.shape

torch.Size([2, 6, 3])

In [124]:
import torch.nn as nn

class CausalAttention(nn.Module):
    
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
        
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # x = batch, 2 x 6 x 3
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose
        attn_scores.masked_fill_( # New, _ops are in place
                                 self.mask.bool() [:num_tokens, :num_tokens], -torch.inf) # `:num_tokens` to account for cases where the n
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim = -1)
        attn_weight = self.dropout(attn_weights) # New
        
        context_vec = attn_weight @ values
        return context_vec
    
torch.manual_seed(789)

context_length = batch.shape[1]
dropout = 0.0
ca = CausalAttention(d_in, d_out, context_length, dropout)
ca(batch)

tensor([[[ 0.3147, -0.4016],
         [ 0.1452, -0.4234],
         [ 0.0916, -0.4243],
         [ 0.0410, -0.3859],
         [ 0.0770, -0.2978],
         [ 0.0277, -0.3262]],

        [[ 0.3147, -0.4016],
         [ 0.1452, -0.4234],
         [ 0.0916, -0.4243],
         [ 0.0410, -0.3859],
         [ 0.0770, -0.2978],
         [ 0.0277, -0.3262]]], grad_fn=<UnsafeViewBackward0>)

### 3.6 Extending single-head attention to multi-head attention

#### 3.6.1 Stacking multiple single-head attention layers

In [126]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads=2, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([
            CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)
        ])
    
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)
    
    
torch.manual_seed(123)

context_length = batch.shape[1]

d_in, d_out = batch.shape[-1], 2

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, dropout=0.0, num_heads=2)
mha(batch)

tensor([[[-0.5740,  0.2727, -0.3132, -0.2272],
         [-0.7272,  0.1840, -0.2252,  0.0507],
         [-0.7733,  0.1575, -0.2013,  0.1339],
         [-0.7002,  0.1201, -0.1638,  0.1384],
         [-0.6551,  0.1314, -0.1673,  0.1825],
         [-0.6447,  0.1017, -0.1410,  0.1740]],

        [[-0.5740,  0.2727, -0.3132, -0.2272],
         [-0.7272,  0.1840, -0.2252,  0.0507],
         [-0.7733,  0.1575, -0.2013,  0.1339],
         [-0.7002,  0.1201, -0.1638,  0.1384],
         [-0.6551,  0.1314, -0.1673,  0.1825],
         [-0.6447,  0.1017, -0.1410,  0.1740]]], grad_fn=<CatBackward0>)

#### 3.6.2 Implementing multi-head attention with weight splits

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads=2, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Dropout(dropout)
        self.out_proj = nn.Linear(d_out, d_out) # Linear layers to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        # We implicitly split the matrix by adding a `num_heads` dimension 
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim) 
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        
        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)
        
        # Compute scaled dot-product attention (aka self-attention) with causal mask
        attn_score = queries @ keys.transpose(2, 3) # Dot product for eaxh head
        
        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        
        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
    
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = attn_weights @ values
        
        # Combine heads, where self.d_out = self. num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection
        
        return context_vec
    
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 4 
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[ 0.3653,  0.3094, -0.0353, -0.7111],
         [ 0.2501,  0.2514, -0.0313, -0.5346],
         [ 0.2187,  0.2477, -0.0308, -0.4916],
         [-0.2933,  0.2398, -0.1027, -0.0831],
         [-0.3759,  0.1794, -0.0977,  0.0578],
         [-0.3243,  0.1925, -0.0927,  0.0050]],

        [[ 0.3653,  0.3094, -0.0353, -0.7111],
         [ 0.2501,  0.2514, -0.0313, -0.5346],
         [ 0.2187,  0.2477, -0.0308, -0.4916],
         [-0.2933,  0.2398, -0.1027, -0.0831],
         [-0.3759,  0.1794, -0.0977,  0.0578],
         [-0.3243,  0.1925, -0.0927,  0.0050]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])
